# Patient Vitals Disease Prediction (Random Forest)

This notebook builds a supervised ML model to predict medical conditions based on patient vital signs. It replaces hardcoded range checks with a data-driven classifier.

**Dataset:** `Synthetic_patient-HealthCare-Monitoring_dataset.csv`

## 1. Import Libraries

In [ ]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Persistence
import joblib

## 2. Load Dataset

In [ ]:
# Load the dataset
data_path = 'Synthetic_patient-HealthCare-Monitoring_dataset.csv'
df = pd.read_csv(data_path)

# Quick overview
df.head()

## 3. Data Preprocessing

In [ ]:
# Columns to drop (irrelevant or alerts)
drop_cols = [
    'Patient Number',
    'Data Accuracy (%)',
    'Heart Rate Alert',
    'SpO₂ Level Alert',
    'Blood Pressure Alert',
    'Temperature Alert',
    'Fall Detection'  # Explicitly dropped per requirement
]

# Drop columns if they exist
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Separate features and target
target_col = 'Predicted Disease'
X = df.drop(columns=[target_col])
y = df[target_col]

# Encode the target labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
# 1. Identify categorical columns (strings)
categorical_cols = X.select_dtypes(include=['object']).columns

# 2. Convert categorical features to numbers (One-Hot Encoding)
# This creates new columns for each category (e.g., Status_NORMAL, Status_HIGH)
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# NOW you can proceed to your existing split and scaling code
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Preserve feature names for later plots
feature_names = X.columns.tolist()
feature_names

## 4. Model Training (Random Forest)

In [ ]:
# Initialize and train the model
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train)

## 5. Evaluation

In [ ]:
# Predictions
y_pred = rf_model.predict(X_test_scaled)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.4f}')

# Classification report
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_,
)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
importances = rf_model.feature_importances_
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=importance_df, x='Importance', y='Feature', palette='viridis')
plt.title('Feature Importance - Vital Signs')
plt.tight_layout()
plt.show()

## 6. Real-world Prediction Function

In [ ]:
def predict_condition(heart_rate, spo2, systolic, diastolic, temp):
    """
    Predict medical condition from raw vitals.
    Inputs are numeric and will be scaled using the trained scaler.
    Returns the human-readable disease label.
    """
    # Arrange inputs in the same order as training features
    input_data = pd.DataFrame([{
        'Heart Rate (bpm)': heart_rate,
        'SpO₂ Level (%)': spo2,
        'Systolic Blood Pressure (mmHg)': systolic,
        'Diastolic Blood Pressure (mmHg)': diastolic,
        'Body Temperature (°C)': temp,
    }])

    # Align with the original training feature order
    input_data = input_data[feature_names]

    # Scale and predict
    scaled = scaler.transform(input_data)
    pred_encoded = rf_model.predict(scaled)[0]
    pred_label = label_encoder.inverse_transform([pred_encoded])[0]
    return pred_label

# Example usage (replace with real values)
# print(predict_condition(82, 97, 120, 78, 36.8))

## 7. Save Model and Scaler

In [ ]:
# Save the trained model and scaler for deployment
joblib.dump(rf_model, 'vitals_random_forest_model.pkl')
joblib.dump(scaler, 'vitals_feature_scaler.pkl')
joblib.dump(label_encoder, 'vitals_label_encoder.pkl')

print('Model, scaler, and label encoder saved successfully.')